# Report 2025 by channel group

*Data Analysis in Python · Topic 13.1*

Your team wants 2025's purchases and revenue by channel group, straight from the GA4 export. Query Simmer Store's export in BigQuery from your own sandbox project.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. In the parameters cell, set `PROJECT_ID` to your own Google Cloud project's ID, as in the video. Run the parameters cell, then the login cell. Choose the Google account you use for Colab, and **Allow**.
2. In the cell marked ✏️, write the query `sql`: from `TABLE`, the `purchase` events from `START_DATE` to `END_DATE`, grouped by the channel group in `session_traffic_source_last_click.cross_channel_campaign.default_channel_group`. Name the columns `channel_group`, `purchases` (the number of purchases) and `revenue` (the sum of `ecommerce.purchase_revenue`, rounded to 2 decimals), and sort by revenue, largest first. Then create the `channels_2025` table: the query's result.
3. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `channels_2025` table has 9 channel groups, with 38,502 purchases and \$3,407,991.38 of revenue. Organic Search has the most revenue, \$868,711.93.

## Parameters

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.colab import auth

PROJECT_ID = ""  # Your own Google Cloud project's ID: the queries run there
TABLE = "simmer-python.simmer_store.events_*"  # Simmer Store's GA4 export, one table per day
START_DATE, END_DATE = "20250101", "20251231"  # _TABLE_SUFFIX values: the tables' dates as text

In [ ]:
# Log in to Google as yourself. Run it, choose your account and Allow.
auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID)

## The query

In [ ]:
# ✏️ Create sql and channels_2025


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    message = f"{type(error).__name__}: {error}" if error is not None else ""
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif "bigquery.jobs.create" in message:
        results.append((None, "BigQuery answered 403: your queries can't run in this project. Set PROJECT_ID to your own project's ID, not simmer-python, then choose Runtime → Restart session and run all."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    project = found.get("PROJECT_ID")
    if not isinstance(project, str) or not project.strip():
        results.append((False, "PROJECT_ID is empty. Set it to your own Google Cloud project's ID, from the project picker in the BigQuery console."))

    table = found.get("channels_2025")
    columns = ["channel_group", "purchases", "revenue"]
    if not isinstance(table, pd.DataFrame) or [column for column in columns if column not in table.columns]:
        results.append((False, f"channels_2025 should be a DataFrame with the columns {', '.join(columns)}: client.query(sql).to_dataframe()."))
    else:
        purchases, revenue = int(table["purchases"].sum()), float(table["revenue"].sum())
        if purchases > 100000:
            results.append((False, f"The table counts {purchases:,} events, not purchases. Keep only the purchases: WHERE ... AND event_name = 'purchase'."))
        elif len(table) != 9:
            results.append((False, f"The channels_2025 table should have 9 rows, one per channel group. It has {len(table)}. Check GROUP BY."))
        elif purchases != 38502 and purchases in (26604, 65106):
            results.append((False, f"The table has {purchases:,} purchases, but 2025 has 38,502. Check that _TABLE_SUFFIX is BETWEEN START_DATE AND END_DATE."))
        elif purchases != 38502:
            results.append((False, f"2025 has 38,502 purchases. The table has {purchases:,}. Check the WHERE clause: the dates and event_name = 'purchase'."))
        elif abs(revenue - 3407991.38) > 0.05:
            results.append((False, f"2025's purchase revenue is $3,407,991.38. The table has ${revenue:,.2f}. Sum ecommerce.purchase_revenue: event_value_in_usd also carries the value of other events."))
        elif table.iloc[0]["channel_group"] != "Organic Search":
            results.append((False, f"Sort by revenue, largest first: ORDER BY revenue DESC. The first row is {table.iloc[0]['channel_group']}."))
        else:
            results.append((True, "9 channel groups, with 38,502 purchases and $3,407,991.38 of revenue. Organic Search has the most, $868,711.93."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
sql = f"""
SELECT
  session_traffic_source_last_click.cross_channel_campaign.default_channel_group AS channel_group,
  COUNT(*) AS purchases,
  ROUND(SUM(ecommerce.purchase_revenue), 2) AS revenue
FROM `{TABLE}`
WHERE _TABLE_SUFFIX BETWEEN '{START_DATE}' AND '{END_DATE}' AND event_name = 'purchase'
GROUP BY channel_group
ORDER BY revenue DESC
"""
channels_2025 = client.query(sql).to_dataframe()
channels_2025
```

</details>